In [1]:
import pandas as pd

df = pd.read_csv("results/results.csv")
df.head()

,trial_id,condition,outcome_type,amount,run_number,model,temperature,prompt_version,prompt_text,response_text,parsed_wager,log_wager,valid,refusal_flag,parse_error_type,timestamp,request_id
0,1,realized_loss_large,realized,-3500,2,gpt-4.1-mini,1.0,absolute,You visited this casino on a previous occasion...,100,100.0,4.605170,True,False,NaN,2026-03-08T23:23:52.471154+00:00,req_afcd6e1b8a894d1488a694725f63883d
1,2,paper_neutral,paper,0,2,gpt-4.1-mini,1.0,absolute,You are currently on a casino visit. So far du...,100,100.0,4.605170,True,False,NaN,2026-03-08T23:23:54.065569+00:00,req_a321ecd4f0104ddb9dfe430d077ba870
2,3,realized_neutral,realized,-30,2,gpt-4.1-mini,1.0,absolute,You visited this casino on a previous occasion...,50,50.0,3.912023,True,False,NaN,2026-03-08T23:23:55.718712+00:00,req_82684b0dfdaa4d49b0a23ffbaaed3e0f
3,4,realized_gain,realized,200,2,gpt-4.1-mini,1.0,absolute,You visited this casino on a previous occasion...,50,50.0,3.912023,True,False,NaN,2026-03-08T23:23:57.172965+00:00,req_2e8b9520bcdd4f80a0613073d1598b91
4,5,paper_gain_large,paper,150,2,gpt-4.1-mini,1.0,absolute,You are currently on a casino visit. So far du...,100,100.0,4.605170,True,False,NaN,2026-03-08T23:23:58.058114+00:00,req_5310e9781e45404db99f235a0d6abbeb


In [2]:
# Drop invalid rows
df = df[df["valid"] == True]

# Convert to numeric (just in case)
df["parsed_wager"] = pd.to_numeric(df["parsed_wager"], errors="coerce")

# Optional: log transform (you already computed it but just in case)
import numpy as np
df["log_wager"] = np.log(df["parsed_wager"])

In [3]:
# Behavioral indicators
df["is_realized"] = (df["outcome_type"] == "realized").astype(int)
df["is_loss"] = (df["amount"] < 0).astype(int)
df["is_gain"] = (df["amount"] > 0).astype(int)

# Magnitude
df["abs_amount"] = df["amount"].abs()

In [4]:
pip install statsmodels


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
df["realized_loss"] = df["is_realized"] * df["is_loss"]

X = df[["is_realized", "is_loss", "realized_loss", "abs_amount"]]
X = sm.add_constant(X)
y = df["parsed_wager"]

model = sm.OLS(y, X).fit()

print(model.summary())

NameError: name 'sm' is not defined

In [6]:
import statsmodels.api as sm

In [7]:
df["realized_loss"] = df["is_realized"] * df["is_loss"]

X = df[["is_realized", "is_loss", "realized_loss", "abs_amount"]]
X = sm.add_constant(X)
y = df["parsed_wager"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           parsed_wager   R-squared:                       0.051
Model:                            OLS   Adj. R-squared:                  0.050
Method:                 Least Squares   F-statistic:                     148.4
Date:                Tue, 24 Mar 2026   Prob (F-statistic):          6.94e-124
Time:                        13:36:54   Log-Likelihood:                -61970.
No. Observations:               11142   AIC:                         1.239e+05
Df Residuals:                   11137   BIC:                         1.240e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            94.0261      1.091     86.188

In [8]:
# Convert model to categorical
df["model"] = df["model"].astype("category")

import statsmodels.formula.api as smf

model = smf.ols(
    "parsed_wager ~ is_realized * is_loss + abs_amount + C(model)",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           parsed_wager   R-squared:                       0.349
Model:                            OLS   Adj. R-squared:                  0.349
Method:                 Least Squares   F-statistic:                     597.8
Date:                Tue, 24 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:37:01   Log-Likelihood:                -59864.
No. Observations:               11142   AIC:                         1.198e+05
Df Residuals:                   11131   BIC:                         1.198e+05
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------

In [9]:
model = smf.ols(
    "parsed_wager ~ is_realized * is_loss * C(model) + abs_amount",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           parsed_wager   R-squared:                       0.386
Model:                            OLS   Adj. R-squared:                  0.385
Method:                 Least Squares   F-statistic:                     249.9
Date:                Tue, 24 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:37:06   Log-Likelihood:                -59538.
No. Observations:               11142   AIC:                         1.191e+05
Df Residuals:                   11113   BIC:                         1.193e+05
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                                                           coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------